# MSB Baseball Detector — YOLOv8 Training (Colab)

Train a single-class YOLOv8 detector for baseball tracking in  
*Mario Superstar Baseball* (Dolphin emulator).

## Setup
1. **Runtime → Change runtime type → GPU** (T4 or better)
2. Upload `yolo_dataset.zip` to Google Drive at `MyDrive/msb/yolo_dataset.zip`  
   OR use the file upload fallback in the cells below.
3. **Run All** cells in order.
4. Trained weights are saved back to Google Drive.

## 1. GPU Check

In [ ]:
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {vram / 1e9:.1f} GB")
else:
    raise RuntimeError(
        "No GPU detected! Go to Runtime → Change runtime type → GPU"
    )

## 2. Install Dependencies

In [ ]:
!pip install -q ultralytics

In [ ]:
import ultralytics
ultralytics.checks()
print(f"Ultralytics {ultralytics.__version__}")

## 3. Dataset Provisioning

**Option A (recommended):** Mount Google Drive and read `yolo_dataset.zip`  
**Option B (fallback):** Upload the zip directly via Colab file upload

In [ ]:
import os
from pathlib import Path

DATASET_DIR = Path("/content/yolo_dataset")
DRIVE_ZIP = Path("/content/drive/MyDrive/msb/yolo_dataset.zip")
OUTPUT_DIR = Path("/content/drive/MyDrive/msb/training_runs")

# Option A: Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    USE_DRIVE = True
    print("Google Drive mounted.")
except Exception as e:
    USE_DRIVE = False
    print(f"Drive mount failed ({e}). Will use file upload instead.")

In [ ]:
import zipfile

zip_path = None

if USE_DRIVE and DRIVE_ZIP.exists():
    zip_path = DRIVE_ZIP
    print(f"Found dataset on Drive: {zip_path}")
else:
    # Option B: File upload fallback
    print("Upload yolo_dataset.zip:")
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            zip_path = Path(f"/content/{name}")
            break

if zip_path is None:
    raise FileNotFoundError("No dataset zip found. Upload yolo_dataset.zip.")

# Unzip
if DATASET_DIR.exists():
    import shutil
    shutil.rmtree(DATASET_DIR)

print(f"Extracting {zip_path} → /content/ ...")
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content/")

assert DATASET_DIR.exists(), f"Expected {DATASET_DIR} after extraction"
print(f"Dataset ready: {DATASET_DIR}")
!ls -la {DATASET_DIR}

In [ ]:
# Verify dataset structure
for split in ("train", "val", "test"):
    imgs = list((DATASET_DIR / "images" / split).glob("*"))
    lbls = list((DATASET_DIR / "labels" / split).glob("*"))
    print(f"  {split}: {len(imgs)} images, {len(lbls)} labels")

# Fix dataset.yaml path to point to /content/yolo_dataset
# The export uses "path: ." (relative); rewrite to absolute for Colab.
yaml_path = DATASET_DIR / "dataset.yaml"
yaml_text = yaml_path.read_text()
import re
yaml_text = re.sub(
    r"^path:.*$",
    f"path: {DATASET_DIR.as_posix()}",
    yaml_text,
    flags=re.MULTILINE,
)
yaml_path.write_text(yaml_text)
print(f"\nUpdated dataset.yaml path → {DATASET_DIR}")
print(yaml_text)

## 4. Training Configuration

Adjust these parameters as needed. Defaults are optimized for  
small-object detection (baseball is tiny in the frame).

In [ ]:
# TRAINING CONFIG
MODEL       = "yolov8n.pt"     # Base model (n=nano, s=small, m=medium)
IMGSZ       = 960              # Higher = better for small objects
EPOCHS      = 150              # Adjust based on dataset size
BATCH       = -1               # Auto-fit to available VRAM
PATIENCE    = 30               # Early stopping patience
SEED        = 42
PROJECT     = "/content/runs"  # Training output directory
NAME        = "ball_detect"    # Run name

## 5. Train

In [ ]:
from ultralytics import YOLO
import time

model = YOLO(MODEL)

t0 = time.time()
results = model.train(
    data=str(yaml_path),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    device=0,
    project=PROJECT,
    name=NAME,
    seed=SEED,
    patience=PATIENCE,
    exist_ok=True,
    # Augmentation tuned for small objects (10-15px baseball)
    mosaic=0.5,      # Reduced from 1.0 — full mosaic shrinks ball to ~7px
    mixup=0.1,
    scale=0.5,
    translate=0.2,
    fliplr=0.0,      # Don't flip — trajectory direction matters
    flipud=0.0,
    hsv_h=0.015,
    hsv_s=0.3,
    hsv_v=0.3,
    close_mosaic=15,  # Disable mosaic for last 15 epochs (convergence)
    verbose=True,
)
duration = time.time() - t0
print(f"\nTraining completed in {duration/60:.1f} minutes")

## 6. Evaluate

In [ ]:
# Find the run directory
RUN_DIR = Path(PROJECT) / NAME
if not RUN_DIR.exists():
    # Ultralytics may append numbers
    candidates = sorted(Path(PROJECT).glob(f"{NAME}*"))
    RUN_DIR = candidates[-1] if candidates else RUN_DIR

print(f"Run directory: {RUN_DIR}")

# Show results
best_pt = RUN_DIR / "weights" / "best.pt"
print(f"Best weights: {best_pt} (exists={best_pt.exists()})")

# Display training curves
from IPython.display import Image, display
results_img = RUN_DIR / "results.png"
if results_img.exists():
    display(Image(filename=str(results_img), width=800))

# Confusion matrix
cm_img = RUN_DIR / "confusion_matrix.png"
if cm_img.exists():
    display(Image(filename=str(cm_img), width=500))

# PR curve
pr_img = RUN_DIR / "PR_curve.png"
if pr_img.exists():
    display(Image(filename=str(pr_img), width=500))

In [ ]:
# Run validation on test set
val_model = YOLO(str(best_pt))
metrics = val_model.val(data=str(yaml_path), split="test")
print(f"\nTest mAP50: {metrics.box.map50:.4f}")
print(f"Test mAP50-95: {metrics.box.map:.4f}")

## 7. Export (Optional: ONNX)

Export to ONNX for faster inference without ultralytics dependency.

In [ ]:
# Uncomment to export ONNX:
# onnx_model = YOLO(str(best_pt))
# onnx_model.export(format="onnx", imgsz=IMGSZ, simplify=True)
# print("ONNX exported.")

## 8. Save to Google Drive

Copy all training artifacts to Drive so nothing is lost when the runtime resets.

In [ ]:
import shutil
from datetime import datetime

if USE_DRIVE:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    dst = OUTPUT_DIR / run_id
    
    print(f"Copying artifacts to Drive: {dst}")
    shutil.copytree(str(RUN_DIR), str(dst))
    
    # Also copy best.pt to a canonical location
    canonical_best = OUTPUT_DIR / "ball_best.pt"
    shutil.copy2(str(best_pt), str(canonical_best))
    
    print(f"\n{'='*55}")
    print(f"  SAVED TO DRIVE")
    print(f"{'='*55}")
    print(f"  Run artifacts: {dst}")
    print(f"  Best weights:  {canonical_best}")
    print(f"\n  Files saved:")
    for f in sorted(dst.rglob("*")):
        if f.is_file():
            size = f.stat().st_size / 1024
            print(f"    {f.relative_to(dst)} ({size:.0f} KB)")
    print(f"{'='*55}")
else:
    print("Drive not mounted. Download files manually:")
    print(f"  Best weights: {best_pt}")
    print(f"  Results: {RUN_DIR / 'results.csv'}")
    # Offer download
    from google.colab import files
    if best_pt.exists():
        files.download(str(best_pt))

## 9. Next Steps

After training is done, bring artifacts back to your local repo:

```bash
# Download from Drive:
#   MyDrive/msb/training_runs/<run_id>/
#   MyDrive/msb/training_runs/ball_best.pt

# Pull into repo:
python tools/colab_package_and_pull.py --pull-artifacts <downloaded_folder>

# Test inference:
python track_folder.py <pitch_folder> --model weights/ball_best.pt
python track_live.py --model weights/ball_best.pt

# Validate:
python validate_tracking.py <pitch_folder> --model weights/ball_best.pt
```